# Experiment 6 — Text Processing Models
This notebook implements the steps shown in the provided PDF: preprocessing, bigrams/trigrams, TF-IDF, Word2Vec, GloVe, BERT embeddings, and PCA visualization.

**Recommended:** Run this in Google Colab. Use **Runtime → Change runtime type → T4 GPU** if available (CPU also works, but BERT may be slower). Run cells from top to bottom.

In [ ]:
%pip -q install gensim transformers seaborn nltk scikit-learn pandas matplotlib

In [ ]:
import re
import numpy as np
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.datasets import fetch_20newsgroups
from IPython.display import display

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

newsgroups_train = fetch_20newsgroups(
    subset='train', remove=('headers', 'footers', 'quotes')
)
df = pd.DataFrame({'data': newsgroups_train.data, 'target': newsgroups_train.target})
display(df.head())

In [ ]:
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = word_tokenize(text)
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words]
    return ' '.join(tokens)

df['processed_text'] = df['data'].apply(preprocess_text)
display(df[['data', 'processed_text']].head())

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

bigram_vectorizer = CountVectorizer(ngram_range=(2, 2))
bigram_matrix = bigram_vectorizer.fit_transform(df['processed_text'])
bigram_feature_names = bigram_vectorizer.get_feature_names_out()
print('Bigram features (first 20):')
print(bigram_feature_names[:20])

trigram_vectorizer = CountVectorizer(ngram_range=(3, 3))
trigram_matrix = trigram_vectorizer.fit_transform(df['processed_text'])
trigram_feature_names = trigram_vectorizer.get_feature_names_out()
print('\nTrigram features (first 20):')
print(trigram_feature_names[:20])

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(df['processed_text'])
feature_names = tfidf_vectorizer.get_feature_names_out()
print('Shape of TF-IDF matrix:', tfidf_matrix.shape)
print('\nSample TF-IDF vectors:')
for i in range(3):
    print(f'\nDocument {i+1}:')
    doc_vector = tfidf_matrix[i]
    nonzero_indices = doc_vector.indices
    values = doc_vector.data
    sample_features = [(feature_names[j], float(v)) for j, v in zip(nonzero_indices, values)]
    sample_features.sort(key=lambda x: x[1], reverse=True)
    print(sample_features[:10])

In [ ]:
from gensim.models import Word2Vec

tokenized_text = [text.split() for text in df['processed_text']]
word2vec_model = Word2Vec(
    sentences=tokenized_text, vector_size=100, window=5,
    min_count=5, workers=4, sg=1, seed=42
)

for word_to_find_similar in ['computer', 'science']:
    if word_to_find_similar in word2vec_model.wv:
        print(f'\nWords similar to {word_to_find_similar!r} (Word2Vec):')
        for word, similarity in word2vec_model.wv.most_similar(word_to_find_similar):
            print(f'  {word}: {similarity:.4f}')
    else:
        print(f'{word_to_find_similar!r} not found in the vocabulary.')

In [ ]:
import gensim.downloader as api

glove_model = api.load('glove-wiki-gigaword-100')  # Downloads about 128 MB; needs internet
print('GloVe model loaded successfully.')
for word_to_find_similar in ['computer', 'science']:
    if word_to_find_similar in glove_model:
        print(f'\nWords similar to {word_to_find_similar!r} (GloVe):')
        for word, similarity in glove_model.most_similar(word_to_find_similar):
            print(f'  {word}: {similarity:.4f}')
    else:
        print(f'{word_to_find_similar!r} not found in the GloVe vocabulary.')

In [ ]:
import torch
from transformers import BertModel, BertTokenizer

model_name = 'bert-base-uncased'
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertModel.from_pretrained(model_name)

def get_bert_embeddings(texts, model, tokenizer, batch_size=16):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    model.eval()
    embeddings = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i + batch_size]
            encoded_input = tokenizer(
                batch_texts, padding=True, truncation=True,
                return_tensors='pt', max_length=128
            )
            input_ids = encoded_input['input_ids'].to(device)
            attention_mask = encoded_input['attention_mask'].to(device)
            outputs = model(input_ids, attention_mask=attention_mask)
            batch_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            embeddings.append(batch_embeddings)
    return np.concatenate(embeddings, axis=0)

# Use the same sampled indices for texts and labels so the plot labels align correctly.
sample_size = 1000
sample_indices = df.sample(n=sample_size, random_state=42).index
sample_texts = df.loc[sample_indices, 'processed_text'].tolist()
bert_embeddings = get_bert_embeddings(sample_texts, model, tokenizer)
print('Shape of BERT embeddings:', bert_embeddings.shape)

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns

pca = PCA(n_components=2, random_state=42)
bert_pca = pca.fit_transform(bert_embeddings)
pca_df = pd.DataFrame(bert_pca, columns=['PCA1', 'PCA2'])
pca_df['target'] = df.loc[sample_indices, 'target'].to_numpy()

plt.figure(figsize=(12, 8))
sns.scatterplot(
    x='PCA1', y='PCA2', hue='target', data=pca_df,
    palette='viridis', legend='full', alpha=0.6, s=25
)
plt.title('PCA of BERT Embeddings (Sampled Data)')
plt.xlabel('PCA Component 1')
plt.ylabel('PCA Component 2')
plt.legend(title='Target', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.grid(True)
plt.tight_layout()
plt.show()

## Notes
- The first run downloads the 20 Newsgroups dataset, the GloVe vectors, and the BERT model; keep the runtime connected.
- If you only need to demonstrate the core experiment quickly, run preprocessing, n-grams, TF-IDF, Word2Vec, and BERT/PCA. The GloVe cell is optional if its 128 MB download is taking too long.
- If a cell fails after a runtime reset, run the notebook from the first cell again.